# OCPP — pengisi daya EV dan CSMS-nya

CSMS kecil dan pengisi simulasi berbagi listener dalam memori, jadi semuanya berjalan di sini tanpa jaringan. Arahkan pengisi ke CSMS sungguhan dengan `UseCsms("wss://host/ocpp/")`, atau layani pengisi sungguhan dengan `OcppCsms` dan `UseTcp(IPAddress.Any, 8887)`.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.28.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.28.0-preview.1"

## Frame RPC
Setiap pesan OCPP adalah array JSON. Penerima yang tidak dapat membacanya menjawab dengan kode galat khusus versi.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Ocpp;
using IoTCom.Net.Transports;

var call = OcppFrame.Parse("[2,\"19223201\",\"BootNotification\",{\"chargePointVendor\":\"VendorX\",\"chargePointModel\":\"SingleSocketCharger\"}]");
Console.WriteLine(call.Describe());
foreach (var f in OcppFrame.DescribeFields(call.ToJson())) Console.WriteLine($"{f.Name,-14} {f.Value}");
foreach (var bad in new[] { "[2,5,\"Heartbeat\",{}]", "[2,\"4\",\"Heartbeat\",[]]" })
    foreach (var v in new[] { OcppVersion.Ocpp16, OcppVersion.Ocpp201 })
        if (!OcppFrame.TryParse(bad, v, out _, out var e)) Console.WriteLine($"{v}: {bad} -> {e.ErrorCode}");

## Sesi pengisian di OCPP 1.6J
Pengisi melakukan boot, mobil dicolokkan, kartu ditempelkan, dan setengah jam pengisian menghasilkan meter value.

In [ ]:
var csmsListener = new InMemoryTransportListener("csms");
var csms = OcppCsms.Create(o => { o.ListenInMemory(csmsListener); o.AuthorizedTags = new HashSet<string> { "04A1B2C3", "APP-7731" }; });
csms.TransactionStopped += (st, tx) => Console.WriteLine($"stopped: {tx}");
await csms.StartAsync();
var ac = OcppChargePointSimulator.Create(o => { o.Identity = "AC-01"; o.UseTransport(csmsListener.Connect); });
await ac.StartAsync();
Console.WriteLine(csms.Stations["AC-01"]);
await ac.PlugInAsync(1, socPercent: 30, batteryKWh: 60, evMaxKW: 11);
Console.WriteLine($"card 04A1B2C3: {await ac.PresentCardAsync(1, "04A1B2C3")}, card DEADBEEF: {await ac.PresentCardAsync(2, "DEADBEEF")}");
await ac.StepAsync(TimeSpan.FromMinutes(30));
await Task.Delay(100);
var bay = csms.Stations["AC-01"].Connectors[1];
Console.WriteLine($"{bay.Status}: {bay.PowerW / 1000:0.0} kW, {bay.Transaction!.EnergyWh / 1000:0.00} kWh, SoC {bay.SoC:0} %");
await ac.UnplugAsync(1);
await Task.Delay(100);

## Perintah di OCPP 2.0.1 harus diizinkan
CSMS menolak perintah sampai diizinkan; lalu CSMS memulai sesi dari jauh, membatasi daya dengan charging profile, dan menghentikannya.

In [ ]:
var dc = OcppChargePointSimulator.Create(o => { o.Identity = "DC-01"; o.Version = OcppVersion.Ocpp201; o.Connectors = 1; o.MaxPowerW = 60_000; o.UseTransport(csmsListener.Connect); });
await dc.StartAsync();
await dc.PlugInAsync(1, socPercent: 18, batteryKWh: 77, evMaxKW: 150);
try { await csms.RemoteStartAsync("DC-01", 1, "APP-7731"); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
csms.ReadOnly = false;
Console.WriteLine($"remote start: {await csms.RemoteStartAsync("DC-01", 1, "APP-7731")}");
await Task.Delay(200);
Console.WriteLine($"limit 30 kW: {await csms.SetPowerLimitAsync("DC-01", 1, 30_000)}");
await dc.StepAsync(TimeSpan.FromMinutes(10));
Console.WriteLine($"charging at {dc.Connectors[0].PowerW / 1000:0.0} kW, SoC {dc.Connectors[0].SoC:0} %");
foreach (var c in await csms.ReadConfigurationAsync("DC-01")) Console.WriteLine($"  {c.Key} = {c.Value}");
var tx = csms.Stations["DC-01"].Connectors[1].Transaction!;
Console.WriteLine($"remote stop {tx.Id}: {await csms.RemoteStopAsync("DC-01", tx.Id)}");
await Task.Delay(200);
await dc.DisposeAsync();
await ac.DisposeAsync();
await csms.DisposeAsync();

## Lebih lanjut
`iotcom ocpp demo`, `iotcom ocpp csms --frames`, `iotcom ocpp charger --csms ws://127.0.0.1:8887/ocpp/ --tag 04A1B2C3`, `iotcom ocpp decode '<json>'`, *Hub pengisian EV lewat OCPP* di Gallery, dan sampel OcppCharger. Lihat `docs/id/protocols/ocpp.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*